# Run Analysis: `20260123_113929`

This notebook loads artifacts under `runs/20260123_113929/` and summarizes Stage2→Stage4 outputs.

- Run dir: `workspace/kms/01_15_new_qlib/runs/20260123_113929`
- Main specs: `specs/*.json`
- Main data: `data/*.parquet`


In [1]:
from pathlib import Path
import sys

# If your notebook CWD is `.../01_15_new_qlib/analysis`, project root is the parent.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "analysis" else cwd
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.run_analyzer import (
    load_run,
    parse_run_log_summary,
    stage2_results_df,
    refinement_history_df,
    stage3_passed_combinations_df,
    stage3_combo_summary_df,
    stage4_combinations_df,
    read_parquet_any,
)

RUN_ID = "20260123_113929"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
assert RUN_DIR.exists(), f"Run dir not found: {RUN_DIR}"
RUN_DIR


PosixPath('/home/user/fin/FinAgent_4090/workspace/kms/01_15_new_qlib/runs/20260123_113929')

In [2]:
run = load_run(RUN_DIR)
rp = run["paths"]

log_summary = parse_run_log_summary(run.get("log_text", ""))
log_summary


{'n_stage2_iterations': 0,
 'stage2_iterations': [],
 'stage3_verdict': None,
 'stage4_started': True}

In [3]:
# High-level run metadata
hyp = run.get("hypothesis_flat") or {}
split = run.get("data_split") or {}

print("hypothesis_id:", hyp.get("hypothesis_id"))
print("horizon_days:", hyp.get("horizon_days"))
print("IS:", split.get("in_sample_start"), "~", split.get("in_sample_end"))
print("OOS:", split.get("out_sample_start"), "~", split.get("out_sample_end"))


hypothesis_id: BH_SR_MeanReversion_3D_v1
horizon_days: 3
IS: 2015-01-01 ~ 2020-12-31
OOS: 2021-01-01 ~ 2025-12-31


## Stage2

- `stage2_summary.json` contains per-formula distribution evidence and PASS/FAIL verdict.


In [4]:
s2 = run.get("stage2_summary")
s2_df = stage2_results_df(s2)
print("stage2 overall:", s2.get("overall_verdict"), "pass_rate=", s2.get("pass_rate"))
s2_df.loc[:, ["formula_id", "observation_id", "verdict", "reasoning"]].head(10)


stage2 overall: FAIL pass_rate= 0.75


,formula_id,observation_id,verdict,reasoning
0,downward_movement_zscore,obs_downward_price_movement,PASS,DIR mean decreases from 0.111513 (Q1) to -0.07...
1,downward_movement_decay,obs_downward_price_movement,PASS,DIR mean decreases from 0.063594 (Q1) to -0.03...
2,undervaluation_momentum_rsi,obs_perceived_undervaluation,FAIL,The observation suggests perceived undervaluat...
3,undervaluation_std_ratio,obs_perceived_undervaluation,PASS,VOL mean increases from 339881.03125 (Q1) to 1...
4,buying_activity_volume_sma,obs_increased_buying_activity,FAIL,"The VOL mean shows a non-monotonic pattern, de..."
5,buying_activity_volume_zscore,obs_increased_buying_activity,PASS,VOL mean increases from 447911.78125 (Q1) to 1...
6,liquidity_addition_volume_decay,obs_liquidity_addition,PASS,VOL mean increases from 94202.398438 (Q1) to 2...
7,liquidity_addition_volume_momentum,obs_liquidity_addition,PASS,VOL mean increases from 521974.875 (Q1) to 116...


## Refinement loop

- `refinement_history.json` shows which formulas failed and what feedback got injected.


In [5]:
rh = run.get("refinement_history")
rh_df = refinement_history_df(rh)
rh_df.head(20)


,iteration,n_total,n_passed,n_failed,pass_rate,failed_formulas
0,1,8,5,3,0.625,"[buying_pressure_volume_delta, contrarian_acti..."
1,2,8,7,1,0.875,[perceived_undervaluation_1]
2,3,8,6,2,0.750,"[undervaluation_momentum_rsi, buying_activity_..."


## Stage3

- `stage3_result.json`: final verdict + the 5 qualified combinations passed to Stage4
- `stage3_ticker_details.json`: per-ticker evaluation details (used to recompute pass_rate / S2-ratio improvement)


In [6]:
s3 = run.get("stage3_result") or {}
print("stage3 verdict:", s3.get("overall_verdict"), "pass_rate=", s3.get("pass_rate"))

s3_passed = stage3_passed_combinations_df(s3)
s3_passed


stage3 verdict: PASS pass_rate= 0.5323232323232323


,instance_id,formula_names,combo_key
0,BH_SR_MeanReversion_3D_v1_instance_001,"[downward_movement_zscore, buying_activity_vol...",buying_activity_volume_zscore|downward_movemen...
1,BH_SR_MeanReversion_3D_v1_instance_002,"[downward_movement_zscore, buying_activity_vol...",buying_activity_volume_zscore|downward_movemen...
2,BH_SR_MeanReversion_3D_v1_instance_003,"[downward_movement_decay, buying_activity_volu...",buying_activity_volume_zscore|downward_movemen...
3,BH_SR_MeanReversion_3D_v1_instance_004,"[downward_movement_decay, buying_activity_volu...",buying_activity_volume_zscore|downward_movemen...


In [7]:
s3_details = run.get("stage3_ticker_details")
s3_combo_summary = stage3_combo_summary_df(s3_details)
s3_combo_summary.head(20)


,combo_key,n_eval,n_pass,pass_rate,avg_confidence,s2_ratio_first,s2_ratio_last,s2_ratio_improvement
1,buying_activity_volume_zscore|downward_movemen...,990,308,0.311111,0.248927,0.505817,0.385787,0.120031
3,buying_activity_volume_zscore|downward_movemen...,990,291,0.293939,0.239415,0.505216,0.421818,0.083398
0,buying_activity_volume_zscore|downward_movemen...,990,239,0.241414,0.187290,0.506167,0.355556,0.150612
2,buying_activity_volume_zscore|downward_movemen...,990,238,0.240404,0.188594,0.505941,0.407619,0.098322


## Stage4

- `stage4_summary.json` has per-combination IS/OOS metrics (Sharpe, returns, drawdown, trades).
- `data/stage4_trades.parquet` has per-trade records.
- `data/stage4_positions.parquet` has daily per-ticker positions (if native Qlib backtest ran).


In [8]:
s4 = run.get("stage4_summary")
s4_df = stage4_combinations_df(s4)
s4_df.loc[:, [
    "combo_idx", "formula_names", "is_sharpe", "oos_sharpe",
    "is_net_return", "oos_net_return", "n_trades", "win_rate", "profit_factor",
]].sort_values("oos_sharpe", ascending=False)


,combo_idx,formula_names,is_sharpe,oos_sharpe,is_net_return,oos_net_return,n_trades,win_rate,profit_factor
2,3,"[downward_movement_decay, buying_activity_volu...",-0.126153,0.579124,-0.428578,0.897594,5361,0.383137,1.130966
0,1,"[downward_movement_zscore, buying_activity_vol...",0.280356,0.279488,0.269814,0.199135,21220,0.375542,1.017529
3,4,"[downward_movement_decay, buying_activity_volu...",0.765659,0.207686,1.942926,0.115679,11912,0.394224,1.093724
1,2,"[downward_movement_zscore, buying_activity_vol...",0.360467,0.042969,0.487250,-0.139490,6648,0.364621,1.018570


In [9]:
trades = read_parquet_any(rp.data_any("stage4_trades.parquet"))
print(trades.shape)

# Exit reasons by combination
trades.groupby(["combo_idx", "exit_reason"]).size().rename("n").reset_index().sort_values(["combo_idx", "n"], ascending=[True, False]).head(30)


(45141, 9)


,combo_idx,exit_reason,n
1,1,time_stop,17561
0,1,stop_loss,2676
2,1,trigger,983
4,2,time_stop,5333
3,2,stop_loss,1050
5,2,trigger,265
7,3,time_stop,4268
6,3,stop_loss,803
8,3,trigger,290
10,4,time_stop,8870


In [10]:
# Trade return distribution (OOS trades are typically tagged the same way; this file includes all trades collected)
trades["return_pct"].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99])


count    45141.000000
mean         0.000793
std          0.048289
min         -0.294697
1%          -0.108634
5%          -0.072172
10%         -0.056543
50%          0.000000
90%          0.056861
95%          0.087619
99%          0.150221
max          0.547442
Name: return_pct, dtype: float64